# 8차시 프로젝트 ② · 제주 감귤 당도 예측하기
> 날씨 데이터로 감귤이 얼마나 달아졌는지 예측한다 — **두 데이터 합치기**와 **여러 특성 회귀**, **교차 검증**에 도전

**프로젝트 흐름** — ① 문제 정의 → ② 데이터 수집 → ③ 탐색·전처리(두 데이터 합치기) → ④ 모델 생성·학습 → ⑤ 모델 평가 → ⑥ 성능 개선(차수·교차 검증·특성 다시 보기) → ⑦ 모델 활용 → ⑧ 보고서

**진행 방법 (PRIMM)** — 🔮 **예측** → ▶️ **실행** → 👀 **관찰** → 🔧 **수정** → ✏️ **직접 만들기** / ✍️ **해석 쓰기**

**사용 데이터** (`data/` 폴더)
| 파일 | 내용 | 출처 |
|---|---|---|
| `seongsan_weather.csv` | 제주 성산 관측소의 일별 기온·강수량·가조시간, 2015-11-15 ~ 2025-11-15 (3,654일) | 기상청 기상자료개방포털 (교재 2부 프로젝트 2, 원본 `성산_기상자료.csv`) |
| `citrus_brix.csv` | 감귤 당도 측정 기록, 2015~2025년 8~12월 약 보름 간격 (91회) | 교재 2부 프로젝트 2 (원본 `감귤_당도측정데이터.csv`) |

## 0. 준비

도구를 불러온다. 코랩에서는 한글 그래프용 `koreanize_matplotlib`이 없으면 자동으로 설치한다.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import koreanize_matplotlib
except ModuleNotFoundError:
    %pip install -q koreanize-matplotlib
    import koreanize_matplotlib

plt.rcParams['figure.figsize'] = (6, 3.5)
DATA = 'data' if os.path.exists('data') else '../data'

---
# ① 문제 정의
**배경** 감귤은 익을수록 달아진다. 농가는 당도를 보고 수확·출하 시기를 정한다. 당도를 재려면 열매를 따서 측정해야 하지만, **날씨 데이터**는 매일 자동으로 쌓인다.

**분석 질문** 그날의 날씨(기온·햇빛)로 감귤의 당도를 예측할 수 있을까?

**머신러닝 문제로 바꾸기**
- 입력 **X** = 최고기온, 최저기온, 가조시간 (특성이 **여러 개**)
- 정답 **y** = 당도 (°Bx, 브릭스 — 과즙 100g에 녹아 있는 당의 g 수)
- 정답이 있고 숫자이므로 → **지도학습 · 회귀**

✍️ **해석 쓰기** — 이 예측을 누가, 언제 쓸 수 있을까? 7차시 흡연 프로젝트와 비교해 X가 어떻게 달라졌는지도 적어 보자.

> (여기에 한두 문장으로 적어 보세요)

---
# ② 데이터 수집
이번에는 **두 개의 데이터**를 쓴다. 기상 자료는 한글 윈도우 형식(`cp949`), 당도 자료는 `utf-8`로 저장되어 있다.

✏️ 빈칸 채우기: 기상 자료를 읽는다.

In [ ]:
weather = pd.read_csv(f'{DATA}/seongsan_weather.csv', encoding='cp949')
print('기상 자료 (행, 열):', weather.___)
weather.head()

▶️ 당도 자료를 읽는다.

In [ ]:
brix = pd.read_csv(f'{DATA}/citrus_brix.csv')
print('당도 자료 (행, 열):', brix.shape)
brix.head()

👀 **관찰** — 두 데이터는 모양이 다르다.
- 기상 자료는 **매일** 기록(3,654일), 당도는 **약 보름에 한 번**(91회)
- 기간도 다르다: 기상은 2015-11-15부터, 당도는 2015-08-03부터
- 두 데이터를 이어 줄 **공통 열**은 `일시`(날짜)뿐이다

---
# ③ 데이터 탐색 및 전처리
## 3-1. 기상 자료 손질

▶️ 모든 행이 같은 값인 `지점`·`지점명` 열을 지우고, `일시`를 날짜 자료형으로 바꿔 인덱스로 정한다. 열 이름의 단위는 짧게 정리한다.

In [ ]:
weather = weather.drop(['지점', '지점명'], axis=1)
weather['일시'] = pd.to_datetime(weather['일시'])
weather = weather.set_index('일시')
weather.columns = ['평균기온', '최저기온', '최고기온', '일강수량', '가조시간']
weather.head(3)

✏️ 빈칸 채우기: 열마다 결측치가 몇 개인지 센다. (3차시)

In [ ]:
weather.isna().___

▶️ 평균기온이 비어 있는 날과 그 앞뒤 날을 본다.

In [ ]:
missing_day = weather[weather['평균기온'].isna()].index[0]
print('평균기온 결측일:', missing_day.date())
weather.loc['2025-05-26':'2025-05-30']

👀 **관찰** — 결측치의 **이유**가 다르다.
- `평균기온` 1개: 그날만 기록이 빠졌다 → 같은 날의 (최저기온 + 최고기온) ÷ 2로 채운다
- `일강수량` 2,070개: 비가 오지 않은 날은 값을 적지 않았다 → **0**으로 채운다

✏️ 빈칸 채우기: 두 가지 방법으로 결측치를 채우고, 남은 결측치의 **전체 합계**를 구한다.

In [ ]:
weather['평균기온'] = weather['평균기온'].fillna((weather['최저기온'] + weather['최고기온']) / 2)
weather = weather.fillna(0)
print('남은 결측치 합계:', weather.isna().sum().___)

✍️ **해석 쓰기** — 결측치를 모두 0으로 채우면 안 되는 이유는? 평균기온 결측을 0으로 채우면 어떤 문제가 생길까?

> (여기에 한두 문장으로 적어 보세요)

▶️ `resample('YE')`로 1년 단위로 묶어 연평균 기온을 본다. 2015년은 11월 15일부터라 낮게 나온다.

In [ ]:
weather['평균기온'].resample('YE').mean().round(2)

✏️ 빈칸 채우기: 연도별 강수량 합계를 막대그래프로 그린다.

In [ ]:
rain = weather['일강수량'].resample('YE').___()
rain.index = rain.index.year
rain.plot(kind='bar', color='#5B8E9F')
plt.title('성산 연 강수량(mm)')
plt.xlabel('연도'); plt.ylabel('mm')
plt.show()

▶️ 기온·가조시간의 분포를 상자 그림으로 본다. 가조시간은 **해가 떠 있는 시간**(낮의 길이)이라 날씨와 상관없이 날짜에 따라 정해진다.

In [ ]:
weather[['평균기온', '최저기온', '최고기온', '가조시간']].plot(kind='box')
plt.title('기온·가조시간 분포')
plt.show()

## 3-2. 당도 자료 손질

▶️ 일시를 날짜로 바꿔 인덱스로 정하고, 기초 통계를 본다.

In [ ]:
brix['일시'] = pd.to_datetime(brix['일시'])
brix = brix.set_index('일시')
brix.describe().round(2)

✏️ 빈칸 채우기: 월별·연도별 평균 당도를 표로 만든다. `unstack()`은 연도를 열로 펼친다. 소수 둘째 자리까지 보이게 하려면?

In [ ]:
pivot = brix.groupby([brix.index.month, brix.index.year])['당도'].mean().unstack()
pivot.___

▶️ 월별 당도 변화를 연도별 선으로 그린다.

In [ ]:
pivot.plot(marker='o', legend=False)
plt.title('월별 당도 변화 (선 하나 = 한 해)')
plt.xlabel('월'); plt.ylabel('당도(°Bx)')
plt.xticks([8, 9, 10, 11, 12])
plt.show()
print('월별 측정 횟수:', brix.groupby(brix.index.month).size().to_dict())

👀 **관찰** — 8월에서 11월로 갈수록 해마다 당도가 오른다. 12월 측정은 2022년 **단 1번**뿐이라 다른 해와 비교할 수 없어 분석에서 뺀다.

▶️ 12월 측정을 제외한다.

In [ ]:
brix = brix[brix.index.month != 12]
print('남은 측정 횟수:', len(brix))

## 3-3. 두 데이터 합치기 (merge)
같은 날짜끼리 이어 붙인다. `how='inner'`는 **양쪽 모두에 있는 날짜만** 남긴다.

✏️ 빈칸 채우기: 날짜 인덱스를 기준으로 두 표를 합치고 크기를 확인한다.

In [ ]:
merged = weather.merge(brix, left_index=True, right_index=True, how='inner')
print('합친 표 (행, 열):', merged.___)
merged.head()

✍️ **해석 쓰기** — 당도 측정은 12월을 빼면 90회인데, 합친 표는 83행이다. 7행은 왜 사라졌을까? (기상 자료가 언제부터인지 떠올려 보자)

> (여기에 한두 문장으로 적어 보세요)

▶️ 상관계수 히트맵으로 당도와 관계가 깊은 특성을 찾는다. (4·5차시)

In [ ]:
sns.heatmap(merged.corr(), annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1)
plt.title('기상 요소와 당도의 상관계수')
plt.show()

✏️ 빈칸 채우기: 상관이 가장 강한 가조시간과 당도의 산점도를 그린다.

In [ ]:
plt.scatter(merged['가조시간'], merged[___], alpha=0.7)
plt.title('가조시간과 당도')
plt.xlabel('가조시간(시간)'); plt.ylabel('당도(°Bx)')
plt.show()

✍️ **해석 쓰기** — 기온·가조시간과 당도의 상관계수가 모두 **음수**(약 −0.8)다. '날이 추워지면 감귤이 달아진다'고 해석해도 될까? 8월→11월에 무슨 일이 함께 일어나는지 생각해 보자.

> (여기에 한두 문장으로 적어 보세요)

---
# ④ 모델 생성 및 학습
교재와 같이 상관이 강한 **최고기온·최저기온·가조시간** 세 특성으로 시작한다. (일강수량은 상관이 거의 0이라 뺀다)

▶️ X(특성 3개)와 y를 고르고, 20%를 평가용으로 나눈다.

In [ ]:
from sklearn.model_selection import train_test_split
X = merged[['최고기온', '최저기온', '가조시간']]
y = merged['당도']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print('학습용:', len(X_train), '/ 평가용:', len(X_test))

✏️ 빈칸 채우기: 선형 회귀(특성이 여러 개라 **다중 선형 회귀**)를 학습하고 특성별 기울기를 본다.

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
pd.Series(model.___, index=X.columns).round(3)

✍️ **해석 쓰기** — 가조시간의 기울기 약 −0.62는 '낮이 1시간 짧아지면 당도가 약 0.6 오른다'는 뜻이다. 다른 두 특성보다 기울기가 큰 이유는 무엇일까?

> (여기에 한두 문장으로 적어 보세요)

---
# ⑤ 모델 평가
R²와 함께 **MSE**(평균 제곱 오차, 교재)와 **MAE**(평균 절대 오차)를 본다. MSE는 오차를 제곱해 평균 내므로 큰 오차에 더 민감하고, 단위가 °Bx²라 해석은 MAE가 쉽다.

✏️ 빈칸 채우기: 평가용 데이터로 점수를 매긴다.

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
y_pred = model.predict(X_test)
print('R² :', round(r2_score(y_test, y_pred), 3))
print('MSE:', round(___(y_test, y_pred), 3))
print('MAE:', round(mean_absolute_error(y_test, y_pred), 3), '°Bx')

▶️ 기준선과 비교: 아무것도 배우지 않고 '학습용 평균 당도'로 모두 찍으면?

In [ ]:
baseline = np.full(len(y_test), y_train.mean())
print('평균으로 찍기 MAE:', round(mean_absolute_error(y_test, baseline), 3), '°Bx')

▶️ 실제 당도와 예측 당도를 산점도로 비교한다. 빨간 대각선에 가까울수록 정확하다.

In [ ]:
plt.scatter(y_test, y_pred)
plt.plot([5.5, 11], [5.5, 11], color='red')
plt.title('실제 vs 예측 당도 (평가용 17개)')
plt.xlabel('실제 당도'); plt.ylabel('예측 당도')
plt.show()

---
# ⑥ 모델 성능 개선하기
## 6-1. 곡선으로: 다항 회귀의 차수 바꾸기
`make_pipeline`은 여러 단계를 **조립 라인**처럼 한 줄로 묶는다.
**표준화(StandardScaler) → 다항 특성(PolynomialFeatures) → 선형 회귀** 순서다.
표준화를 먼저 하는 이유: 기온 33을 네제곱하면 약 119만처럼 큰 수가 되어 계산이 불안정해지기 때문이다(7차시 경과연수와 같은 이유).

🔮 예측: 차수를 1→4로 올리면 평가 R²는 계속 오를까? 예측한 뒤 ▶️ 실행한다.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

def poly_model(degree):
    return make_pipeline(StandardScaler(), PolynomialFeatures(degree=degree, include_bias=False), LinearRegression())

rows = []
for d in [1, 2, 3, 4]:
    m = poly_model(d).fit(X_train, y_train)
    rows.append({'차수': d, '특성 개수': m[1].n_output_features_,
                 '학습 R²': round(r2_score(y_train, m.predict(X_train)), 3),
                 '평가 R²': round(r2_score(y_test, m.predict(X_test)), 3),
                 '평가 MSE': round(mean_squared_error(y_test, m.predict(X_test)), 3)})
pd.DataFrame(rows)

✍️ **해석 쓰기** — 특성 3개로 차수를 4까지 올리면 특성이 34개가 된다. 학습 데이터는 66개뿐인데 특성이 많아지면 어떤 일이 생길까? 학습 R²와 평가 R²의 변화로 설명해 보자.

> (여기에 한두 문장으로 적어 보세요)

## 6-2. 운에 맡기지 않기: 교차 검증 (K-Fold)
평가용 17개를 **어떻게 뽑았느냐**에 따라 점수가 크게 달라질 수 있다. **5겹 교차 검증**은 데이터를 5묶음으로 나눠, 묶음마다 한 번씩 돌아가며 시험을 보고 점수를 평균 낸다. 모든 데이터가 한 번씩 평가용이 된다.

✏️ 빈칸 채우기: 5묶음(`n_splits=5`)으로 나눠 1차 모델의 점수 다섯 개를 본다.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
kf = KFold(n_splits=5, shuffle=True, random_state=2)
scores = cross_val_score(poly_model(1), X, y, cv=kf, scoring='r2')
print('묶음별 R²:', scores.round(3))
print('평균 R²  :', round(scores.___, 3))

👀 **관찰** — 같은 모델인데 묶음에 따라 R²가 약 0.47에서 0.91까지 흔들린다. 점수 하나만 믿으면 안 되는 이유다.

🔮 예측 → ▶️ 실행: 차수 1~5를 교차 검증으로 비교한다. 6-1의 결론과 같을까?

In [ ]:
cv_rows = []
for d in range(1, 6):
    r2 = cross_val_score(poly_model(d), X, y, cv=kf, scoring='r2').mean()
    mse = -cross_val_score(poly_model(d), X, y, cv=kf, scoring='neg_mean_squared_error').mean()
    cv_rows.append({'차수': d, '평균 R²': round(r2, 3), '평균 MSE': round(mse, 3)})
pd.DataFrame(cv_rows)

✍️ **해석 쓰기** — 한 번 나눈 평가(6-1)에서는 3차가 가장 좋았지만, 교차 검증에서는 몇 차가 가장 좋았나? 어느 결과를 더 믿어야 할까? 5차의 점수는 왜 저렇게 나왔을까?

> (여기에 한두 문장으로 적어 보세요)

## 6-3. 🔧 탐구: 특성을 다시 생각하기 — 날씨일까, 계절일까?
가조시간은 날씨가 아니라 **날짜로 정해지는 낮의 길이**다. 그렇다면 '날씨'가 아니라 '**계절(날짜)**' 자체가 당도를 설명하고 있는 것은 아닐까? 1월 1일부터 며칠째인지를 나타내는 **일차** 하나만으로 모델을 만들어 비교해 보자.

✏️ 빈칸 채우기: 날짜에서 '1년 중 며칠째'(`dayofyear`)를 꺼내 파생 변수를 만들고, 교차 검증으로 비교한다.

In [ ]:
merged['일차'] = merged.index.___
compare_feats = {'교재 (최고·최저기온·가조시간)': ['최고기온', '최저기온', '가조시간'],
                 '가조시간 하나': ['가조시간'],
                 '일차(날짜) 하나': ['일차']}
for name, cols in compare_feats.items():
    r2 = cross_val_score(poly_model(2), merged[cols], y, cv=kf, scoring='r2').mean()
    print(f'{name:22s} 2차 교차 검증 R²: {r2:.3f}')

✍️ **해석 쓰기** — '날짜 하나'만으로도 날씨 세 가지와 비슷하거나 더 좋은 점수가 나왔다. 이 결과는 '기온이 낮아서 달아진다'는 해석에 대해 무엇을 말해 줄까? (6차시: 상관관계와 인과관계, 숨은 변수)

> (여기에 한두 문장으로 적어 보세요)

---
# ⑦ 모델 활용
교차 검증에서 좋았던 **2차 모델**을 전체 데이터로 다시 학습해 최종 모델로 쓴다.

✏️ 빈칸 채우기: 최종 모델을 전체 X, y로 학습한다.

In [ ]:
final_model = poly_model(2)
final_model.fit(X, y)
print('최종 모델 학습 완료 · 사용한 데이터:', len(___), '개')

▶️ 월별 대표 날씨(측정일 평균)를 넣어 당도를 예측한다.

In [ ]:
scenario = pd.DataFrame({'최고기온': [31.5, 28.2, 24.5, 19.1],
                         '최저기온': [25.6, 23.2, 17.6, 10.2],
                         '가조시간': [13.6, 12.6, 11.6, 10.6]},
                        index=['8월', '9월', '10월', '11월'])
scenario['예측 당도'] = final_model.predict(scenario).round(2)
scenario

✏️ 직접 바꿔 보기: 교재 예시(최고 33.0℃, 최저 25.1℃, 가조 13.8시간)의 당도를 예측한다.

In [ ]:
new_day = pd.DataFrame({'최고기온': [33.0], '최저기온': [25.1], '가조시간': [13.8]})
final_model.___(new_day).round(2)

▶️ 모델을 파일로 저장한다. 이 파일을 대시보드(웹앱)에서 불러오면 날씨를 입력해 당도를 바로 예측할 수 있다.

In [ ]:
import joblib
joblib.dump(final_model, 'brix_model.joblib')
loaded = joblib.load('brix_model.joblib')
print('불러온 모델로 예측:', loaded.predict(new_day).round(2))

✍️ **해석 쓰기** — 이 모델을 감귤 농가에 소개한다면 어떻게 쓰라고 할까? 6-3의 발견(날짜 하나로도 비슷한 성능)을 생각하면, 이 모델로 '올해 날씨가 당도에 어떤 영향을 줬는지'까지 알 수 있을까?

> (여기에 한두 문장으로 적어 보세요)

---
# ✏️ 직접 만들기 — 나만의 특성 조합 찾기
특성 조합을 바꿔 교차 검증 점수를 비교해 보자. 아래 예시를 고치거나 새 조합을 추가한다.

✏️ M1: 특성 조합 목록에 `['평균기온', '가조시간']`과 `['최고기온', '최저기온', '가조시간', '일강수량']`을 넣어 1차·2차 점수를 비교한다.

In [ ]:
my_sets = [['평균기온', '가조시간'],
           ['최고기온', '최저기온', '가조시간', ___]]
for cols in my_sets:
    for d in [1, 2]:
        r2 = cross_val_score(poly_model(d), merged[cols], y, cv=kf, scoring='r2').mean()
        print(cols, f'{d}차 → 평균 R² {r2:.3f}')

✍️ **해석 쓰기** — M1: 일강수량을 넣었더니 점수가 어떻게 되었나? 상관계수 히트맵에서 일강수량과 당도의 상관계수가 얼마였는지와 연결해 설명해 보자.

> (여기에 한두 문장으로 적어 보세요)

✏️ M2: 가장 좋았던 조합으로 최종 모델을 만들고, 내가 정한 날씨(예: 10월 초)로 당도를 예측한다.

In [ ]:
best_cols = ['평균기온', '가조시간']   # 내가 고른 조합으로 바꿔 보기
my_model = poly_model(2).fit(merged[best_cols], y)
my_day = pd.DataFrame({'평균기온': [22.0], '가조시간': [11.8]})   # 내가 정한 날씨
my_model.predict(my_day).round(2)

---
# ⑧ 프로젝트 보고서 정리
| 항목 | 내용 |
|---|---|
| 문제 정의 | 어떤 질문에 답하려 했나? 누가 쓸까? |
| 데이터 | 두 데이터의 출처·기간·간격은? 합친 뒤 몇 행? |
| 전처리 | 결측치를 어떻게, 왜 다르게 채웠나? 무엇을 뺐나? |
| 탐색 결과 | 당도와 가장 관계 깊은 특성은? 상관의 방향은? |
| 모델과 평가 | 어떤 모델이 가장 좋았나? R²·MSE·MAE는? |
| 개선 | 차수·교차 검증·특성 탐구에서 무엇을 알았나? |
| 활용 제안 | 결과를 누구에게 어떻게 쓰자고 할까? |
| 한계 | 데이터 83개 · 측정일 하루의 날씨만 사용 · 계절(숨은 변수) · 한 관측소 · 품종/과수원 정보 없음 |